In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),  
    password=os.getenv("NEO4J_PASSWORD"),
) 

In [6]:
cypher_query = """
create (a:Person {name: 'Alice', age: 30})
return a
"""
graph.query(cypher_query)

[{'a': {'name': 'Alice', 'age': 30}}]

In [10]:
def reset_database(graph):
    """
    reset Database 
    """
    graph.query("MATCH (n) DETACH DELETE n")


  #delete all constraints
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint["name"]
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")

    # delete all indexes
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("Database reset complete.")

reset_database(graph)

Database reset complete.


In [11]:
cypher_query = """
CREATE (a:Person {name: 'Alice', age: 30})
"""
graph.query(cypher_query)

[]

In [12]:
cypher_query = """
CREATE (a:Person {name: 'Heide', age: 30})
CREATE (b:Person {name: 'Bob', age: 25})
CREATE (c:City {name: 'Seoul', population: 9733509})
"""
graph.query(cypher_query)

[]

In [13]:
cypher_query = """
MATCH (a:Person {name: 'Alice'}), (b:Person {name: 'Bob'})
CREATE (a)-[:KNOWS{since: 2020}]->(b)
"""
graph.query(cypher_query)

[]

In [15]:
cypher_query = """
CREATE (a:Person {name: 'Park'})-[:LIVES_IN]->(b:City {name: 'Manchester'})
return a, b
"""
graph.query(cypher_query)

[{'a': {'name': 'Park'}, 'b': {'name': 'Manchester'}}]

In [22]:
cypher_query = """
MATCH (a:Person {name: 'Alice'}),
      (c:City {name: 'Seoul'})
CREATE (a)-[r:LIVES_IN]->(c)
return a,r,c
"""
graph.query(cypher_query)

[{'a': {'name': 'Alice', 'age': 30},
  'r': ({'name': 'Alice', 'age': 30},
   'LIVES_IN',
   {'name': 'Seoul', 'population': 9733509}),
  'c': {'name': 'Seoul', 'population': 9733509}}]

In [25]:
cypher_query = """
MATCH (a:Person {name: 'Alice'}),
(b:Person {name: 'Heide'}),
(c:City {name: 'Seoul'})
CREATE (a)-[r:LIVES_IN]->(c),
        (a)-[r2:KNOWS]->(b),
        (a)<-[r3:KNOWS]-(b)
return a, b, c
"""
graph.query(cypher_query)

[{'a': {'name': 'Alice', 'age': 30},
  'b': {'name': 'Heide', 'age': 30},
  'c': {'name': 'Seoul', 'population': 9733509}}]

In [28]:
cypher_query = """
MATCH (p:Person)

return p
"""
results = graph.query(cypher_query)
for record in results:
    print(record)

{'p': {'name': 'Heide', 'age': 30}}
{'p': {'name': 'Alice', 'age': 30}}
{'p': {'name': 'Bob', 'age': 25}}
{'p': {'name': 'Park'}}


In [30]:
cypher_query = """
MATCH (p:Person)
RETURN p.name AS name
"""
results = graph.query(cypher_query)

for record in results:
    print(record)

{'name': 'Heide'}
{'name': 'Alice'}
{'name': 'Bob'}
{'name': 'Park'}


In [31]:
cypher_query = """
MATCH (p:Person)-[r:LIVES_IN]->(c:City)
RETURN p.name AS person, c.name AS city
"""
graph.query(cypher_query)

[{'person': 'Alice', 'city': 'Seoul'},
 {'person': 'Alice', 'city': 'Seoul'},
 {'person': 'Alice', 'city': 'Seoul'},
 {'person': 'Park', 'city': 'Manchester'}]

In [33]:
cypher_query = """
MATCH (p:Person)-[r:LIVES_IN]->(c:City)
where p.age > 25 and c.name = 'Seoul'
RETURN p
"""
graph.query(cypher_query)

[{'p': {'name': 'Alice', 'age': 30}},
 {'p': {'name': 'Alice', 'age': 30}},
 {'p': {'name': 'Alice', 'age': 30}}]

In [34]:
cypher_query = """
MATCH (p:Person {name:'Park'})

SET p.age = 44

return p
"""  
graph.query(cypher_query)

[{'p': {'name': 'Park', 'age': 44}}]

In [35]:
cypher_query = """
MATCH (p:Person {name: 'Alice'})
DELETE p
"""
graph.query(cypher_query)

ConstraintError: {neo4j_code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Cannot delete node<1>, because it still has relationships. To delete this node, you must first delete its relationships.} {gql_status: G1001} {gql_status_description: error: dependent object error - edges still exist}

In [36]:
cypher_query = """
MATCH (p:Person {name: 'Alice'})
DETACH DELETE p
"""
graph.query(cypher_query)

[]

In [37]:
cypher_query = """
MATCH (p:Person{name:"Park"})-[r:LIVES_IN]->(c:City)
DELETE r
RETURN p, c
"""
graph.query(cypher_query)

[{'p': {'name': 'Park', 'age': 44}, 'c': {'name': 'Manchester'}}]

In [38]:
cypher_query = """
MATCH (p:Person{name:"Park"})
REMOVE p.age
RETURN p
"""
graph.query(cypher_query)

[{'p': {'name': 'Park'}}]

In [41]:
cypher_query = """
MATCH (p:Person {name:'Bob'})
REMOVE p:Person

"""
graph.query(cypher_query)

[]

In [45]:
cypher_query = """
MATCH (p {name: "Bob"})
SET p:Person, p.age = 25
ReTURN p
"""
graph.query(cypher_query)


[{'p': {'name': 'Bob', 'age': 25}}]

In [46]:
cypher_query = """
MATCH (p:Person)
RETURN p
ORDER BY p.age
lIMIT 2 
"""
result = graph.query(cypher_query)
for record in result:
    print(record)


{'p': {'name': 'Bob', 'age': 25}}
{'p': {'name': 'Heide', 'age': 30}}


In [47]:
cypher_query = """
MATCH (p:Person)
RETURN p
ORDER BY p.age
skip 1
lIMIT 1 
"""
result = graph.query(cypher_query)
for record in result:
    print(record)


{'p': {'name': 'Heide', 'age': 30}}
